### Coding Order 

In [0]:
%sql
SELECT DISTINCT TOP 2
COL_1, SUM(COL_2)
FROM TABLE
WHERE COL_ = 10
GROUP BY COL_1
HAVING SUM(COL_2)>2
ORDER BY COL_1 ASC


### Execution Order

In [0]:
FROM (including joins)
WHERE
GROUP BY 
HAVING 
SELECT 
DISTINCT 
ORDER BY
TOP

### index

In [0]:
>>Data Structure provides quick access to data, optimizing the speed of your query.
Type of Indexes: 
    Structure: 
        Clustered Index
        Non - Clustered Index
    Storage:
        Rowstore Index
        Columnstore Index
    Functions:
        Unique Index
        Filtered Index

### Heap Structure

In [0]:
>>A heap in SQL (especially in systems like SQL Server) is the simplest way a table can be stored.
>>How data is stored
    Data is stored in pages (8 KB blocks),pages are not ordered based on any column.
    When you insert data → it goes to the first available free space
    SQL Server uses an internal pointer called a RID (Row Identifier):
    RID = (File ID, Page Number, Slot Number)
>>For Heap structure, "Insert = just find free space → write row"

![image_1774648974999.png](./image_1774648974999.png "image_1774648974999.png")

### Clustered Index

In [0]:
A clustered index defines how the actual data in a table is physically stored and ordered.

A clustered index = the table itself, sorted by a key. The data rows are stored in order of the indexed column.
There is no separate structure like other indexes.
That’s why: A table can have only ONE clustered index

CODE:
    
    CREATE CLUSTERED INDEX idx_emp_id ON employees(id);

In most databases (like SQL Server):
PRIMARY KEY → automatically becomes clustered index (unless specified otherwise)

### B-Tree Clustered Index

In [0]:
A B-Tree clustered index is an index where the table data is physically stored in a sorted B-Tree structure based on the index key. The leaf nodes contain the actual data rows, enabling fast lookups and range queries. Since it defines the data order, a table can have only one clustered index.

![image_1774649435102.png](./image_1774649435102.png "image_1774649435102.png")

### B-Tree Non-Clustered Index

In [0]:
A B-Tree non-clustered index is a separate sorted structure that stores indexed column values along with pointers to the actual table rows. The leaf nodes do not contain full data, only references (RID or clustered key). It improves query performance without changing the physical order of the table.

![image_1774659523923.png](./image_1774659523923.png "image_1774659523923.png")

In [0]:
>>Non cluetserd index doesnot care if the data is shorted or not. So if there is both clustered and non clustered index available, non clustered index will ponit to shorted Data pages.

![image_1774659960407.png](./image_1774659960407.png "image_1774659960407.png")

In [0]:
>>Multiple Columns in a Index: Composite Index

In [0]:
CREATE [CLUSTERED | NONCLUSTERED] INDEX index_name ON table_name (clm_1, clm_2..)

>>IF the type of the index is not defined, SQL will consider as NONCLUSTERED.
>>If we do nor create any Index, SQL by default creates clueterd index on primary ley.

![image_1774660540952.png](./image_1774660540952.png "image_1774660540952.png")

### Rowstore Index and Column Store index

In [0]:
>>By default heap structure uses Rowstore Index.
>>A columnstore index stores data column-wise instead of row-wise, meaning values of each column are stored together. This allows:
High compression
Very fast analytical queries (aggregations like SUM, COUNT)
>>It is ideal for data warehousing/OLAP workloads, but not for frequent row-level inserts/updates.
>>After Final compression in Column store index, Data Stored in LoB page (Large Object Page) instead of Data Page like healp/Rowstore...

![image_1774661179281.png](./image_1774661179281.png "image_1774661179281.png")

![image_1774661706326.png](./image_1774661706326.png "image_1774661706326.png")

![image_1774661742517.png](./image_1774661742517.png "image_1774661742517.png")

In [0]:
In clustered indexes, the physical storage of the table is changed—either to a sorted B-Tree (rowstore) or to a columnar compressed format in case of columnstore. In non-clustered indexes, the base table remains unchanged, and an additional index structure is created on top, storing keys and references to the actual data.

In [0]:
How Column Store is faster

![image_1774665065804.png](./image_1774665065804.png "image_1774665065804.png")

In [0]:
Where to use what..eg Data Lake CCI

![image_1774665437880.png](./image_1774665437880.png "image_1774665437880.png")

![image_1774665534635.png](./image_1774665534635.png "image_1774665534635.png")

![image_1774665642415.png](./image_1774665642415.png "image_1774665642415.png")

![image_1774666263813.png](./image_1774666263813.png "image_1774666263813.png")

![image_1774666229857.png](./image_1774666229857.png "image_1774666229857.png")

In [0]:
UNIQUE INDES SYNTAX

>>This is to restrict the duplicate values 

![image_1774667143607.png](./image_1774667143607.png "image_1774667143607.png")

In [0]:
FILTER INDEX:
    
>>Used to store the indexs where specific data need to filtered. Such as if there are inactive users available and we dont really access those inactive user records, we can index only active users here. This is to save the STORAGE optimization of indexes. 

In [0]:
Syntax

![image_1774667650539.png](./image_1774667650539.png "image_1774667650539.png")

![image_1774669815080.png](./image_1774669815080.png "image_1774669815080.png")

### Partitioning

In [0]:
>>In SQL Server, partitioning is implemented using partition functions and partition schemes. A partition function defines how data is split, while a partition scheme maps partitions to filegroups. Filegroups store data physically via database files, allowing better performance and manageability for large tables.

In [0]:

ALTER DATABASE MyDB ADD FILEGROUP fg_2023;
ALTER DATABASE MyDB ADD FILEGROUP fg_2024;

In [0]:

CREATE PARTITION FUNCTION pf_year (INT)
AS RANGE LEFT FOR VALUES (2023, 2024);

In [0]:
One important clarification

This is SQL Server-specific.

In:

PostgreSQL / MySQL → no filegroups, simpler partition syntax
Delta Lake / Spark → partitioning = folder-based, no filegroups

In [0]:
In Delta Lake, partitioning is implemented by storing data in separate folders based on column values using the PARTITIONED BY clause. It improves performance through partition pruning, where only relevant data is scanned during queries.

In [0]:

CREATE TABLE sales (
    id INT,
    amount DOUBLE,
    sale_date DATE
)
USING DELTA
PARTITIONED BY (sale_date);

### SQL OPTIMIZE

In [0]:
My approach is to reduce data movement and scan size using indexing and partitioning, apply early filtering, and use efficient joins. I rely on execution plans to identify bottlenecks and iteratively tune the query.

In [0]:
SQL query optimization is the process of improving the performance of SQL statements to ensure faster execution and reduced consumption of resources such as CPU, memory, and disk I/O. This process involves analyzing the query execution plan and applying various techniques to fetch only the necessary data efficiently. 

Key techniques for SQL query optimization include:
Use Indexes Effectively: 
    Indexes act as a lookup to help the database engine quickly locate data without scanning the entire table. Create indexes on columns frequently used in WHERE clauses, JOIN conditions, and ORDER BY clauses. Avoid over-indexing, as it can slow down write operations (INSERT, UPDATE, DELETE).
Avoid SELECT *: 
    Explicitly specify only the columns you need in your SELECT statement. Retrieving unnecessary columns increases memory usage, network transfer time, and overall processing load.
Filter Data Early: 
    Use the WHERE clause to filter out unneeded rows as early as possible in the query process. Filtering before performing joins or aggregations significantly reduces the volume of data processed in later stages. The HAVING clause should be used for filtering after a GROUP BY operation, not as a substitute for WHERE.
Optimize JOIN Operations: 
    Use the most efficient JOIN type for your needs, preferring INNER JOIN when only matching records are required. Ensure columns used in join conditions are indexed and have matching data types to avoid slow conversions.
Use EXISTS Instead of IN (for Subqueries): 
    When checking for the mere existence of a record in a subquery, EXISTS is often faster than IN because it stops scanning as soon as it finds the first match.
Avoid Functions on Indexed Columns: 
    Applying functions or calculations to indexed columns in a WHERE clause prevents the database from using the index efficiently, forcing a full table scan.
Inefficient: 
    WHERE YEAR(joining_date) = 2022
Efficient: 
    WHERE joining_date >= '2022-01-01' AND joining_date < '2023-01-01'
Use UNION ALL Over UNION: 
    UNION removes duplicate rows, which requires an extra sorting process, while UNION ALL keeps all rows without the overhead of deduplication. Use UNION ALL unless duplicate removal is a strict requirement.
Break Down Complex Queries: 
    For readability and performance, use Common Table Expressions (CTEs) or temporary tables to break large, complex queries into smaller, logical steps. Temporary tables can be especially useful for storing intermediate results and handling bulk data operations efficiently.
Limit Result Set Size: 
    Use the LIMIT and OFFSET clauses for pagination or data exploration to restrict the number of rows returned, which reduces processing time and network load.
Analyze Query Execution Plans: 
    Use the database's built-in tools (e.g., EXPLAIN in MySQL/PostgreSQL, Execution Plan Viewer in SQL Server) to understand how the optimizer processes your query. This helps identify bottlenecks like full table scans or expensive join operations.
Keep Database Statistics Updated: 
Query optimizers rely on database statistics to estimate the most efficient execution plan. Regularly update statistics to ensure the optimizer makes accurate decisions as data changes.
Avoid Cursors and Loops in SQL: Force the engine to process data in sets (set-based operations) using joins or bulk operations rather than row-by-row processing with cursors or application-side loops

### Covering Index

In [0]:
A covering index is an index that contains all the columns required by a query, so the database can return results without accessing the main table.

CREATE INDEX idx_emp 
ON employees(id) 
INCLUDE (name, salary);

A covering index is an index that includes all the columns needed by a query, allowing the database to retrieve results directly from the index without accessing the base table. This improves performance by avoiding additional lookups.

In [0]:
Apart from standard joins, we also use semi-joins and anti-joins, which are logical operations. A semi-join returns rows that have matches, typically implemented using EXISTS, while an anti-join returns non-matching rows, implemented using NOT EXISTS or LEFT JOIN with NULL filtering.